# LSM-Trees: Forensic Exploration

**Data Mastery Lab** — Salesforce Data Cloud

---

## What we'll explore

1. **The LSM-Tree write path** — WAL → Memtable → SSTable flush (why writes are cheap)
2. **The LSM-Tree read path** — Memtable → SSTables + Bloom filters (why reads pay the cost)
3. **Compaction** — merging SSTables to reclaim space and reduce read amplification
4. **Crash recovery** — reconstructing the memtable from the WAL
5. **B+Tree vs LSM-Tree** — head-to-head comparison of the two fundamental data structures

### The core insight

> An LSM-Tree is an **optimization for writes at the cost of reads**.
> Instead of maintaining a sorted structure on every write (B-tree),
> it buffers writes in memory and sorts lazily on flush.

| Operation | LSM-Tree | B+Tree | Why |
|-----------|----------|--------|-----|
| Write | O(1) amortized | O(log_B N) | Append to log + memtable insert |
| Point read | O(L × log N) | O(log_B N) | Must check memtable + L levels of SSTables |
| Range scan | O(L × N/B) | O(log_B N + K) | Must merge-scan across all levels |
| Space | Higher (amplification) | ~1x | Multiple copies during compaction |

Where L = number of levels, N = total keys, B = block size.

In [ ]:
import sys, os, shutil, time, random, json
sys.path.insert(0, '.')
from lsm_engine import LSMTree, WAL, Memtable, SSTable, BloomFilter
from lsm_viz import render_lsm_state, render_write_path, get_lsm_snapshot
from IPython.display import display, HTML, Image

OUTPUT_DIR = "_output"
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
os.makedirs(OUTPUT_DIR)
print(f"Output directory: {OUTPUT_DIR}/")

---
## Part 1: The Write Path — Why Writes Are Cheap

In a B+Tree, every write must:
1. Traverse the tree to find the right leaf
2. Insert in sorted order
3. Potentially split nodes (cascade up)

In an LSM-Tree, a write is just:
1. **Append** to the WAL (sequential I/O — fast!)
2. **Insert** into the memtable (in-memory sorted structure — fast!)
3. That's it. The data is sorted *later*, when the memtable flushes.

```
Write path:

  put(k, v)  ──→  WAL (append)  ──→  Memtable (sorted in RAM)
                   │                        │
                   │ sequential I/O          │ when full...
                   │ (fast!)                 │
                   ▼                         ▼
              Durability              Flush to SSTable (sorted file on disk)
```

In [ ]:
# Create an LSM-Tree with a small memtable (capacity=10) to see flushes happen quickly
db_path = f"{OUTPUT_DIR}/demo_db"
db = LSMTree(db_path, memtable_capacity=10)

print("Inserting 25 keys with memtable capacity=10...")
print(f"{'Key':<15} {'Memtable Size':>14} {'Flushed?':>9} {'SSTables':>9}")
print("-" * 50)

snapshots = []
for i in range(1, 26):
    key = f"user_{i:04d}"
    value = f'{{"name": "User {i}", "score": {random.randint(1, 100)}}}'
    result = db.put(key, value)
    flush_marker = " << FLUSH" if result['flushed'] else ""
    compact_marker = " + COMPACT" if result['compacted'] else ""
    print(f"{key:<15} {len(db.memtable):>14} {str(result['flushed']):>9} "
          f"{len(db.sstables):>9}{flush_marker}{compact_marker}")
    
    # Capture snapshots at interesting moments
    if i <= 3 or result['flushed'] or result['compacted']:
        snap = get_lsm_snapshot(db)
        event = "flush" if result['flushed'] else ("compact" if result['compacted'] else "insert")
        snapshots.append((key, event, snap))

print(f"\n{db.describe()}")

In [ ]:
# Visualize the LSM-Tree state at key moments
for key, event, snap in snapshots:
    title = f"After put({key}) — {event.upper()}"
    fname = f"write_{key}_{event}"
    highlight = ["wal", "memtable"] if event == "insert" else []
    path = render_lsm_state(
        memtable_keys=snap["memtable_keys"],
        memtable_capacity=snap["memtable_capacity"],
        wal_entries=snap["wal_entries"],
        levels=snap["levels"],
        title=title,
        filename=fname,
        output_dir=OUTPUT_DIR,
        highlight_key=key,
        highlight_path=highlight,
    )
    print(f"\n── {title} ──")
    if path.endswith(".png"):
        display(Image(filename=path))
    else:
        print(f"  (DOT file: {path} — paste into https://dreampuf.github.io/GraphvizOnline/)")

print("\n→ Watch the memtable fill up, then FLUSH to an SSTable on disk.")
print("→ After flush, the memtable is empty and the WAL is reset.")
print("→ SSTables accumulate at Level 0 until compaction merges them.")

In [ ]:
# Let's look at what's actually on disk
print("Files on disk:")
for f in sorted(os.listdir(db_path)):
    fpath = os.path.join(db_path, f)
    size = os.path.getsize(fpath)
    print(f"  {f:<25} {size:>8,} bytes")

# Peek inside an SSTable — it's just sorted JSON lines
sst_files = [f for f in os.listdir(db_path) if f.startswith("sst_")]
if sst_files:
    first_sst = sorted(sst_files)[0]
    print(f"\nContents of {first_sst} (first SSTable):")
    with open(os.path.join(db_path, first_sst)) as f:
        for i, line in enumerate(f):
            entry = json.loads(line)
            print(f"  {entry['key']}: {entry['value'][:50]}...")
            if i >= 4:
                print(f"  ... ({sum(1 for _ in open(os.path.join(db_path, first_sst)))} entries total)")
                break
    print("\n→ Keys are SORTED within each SSTable. This is why it's called Sorted String Table.")
    print("→ Each SSTable is IMMUTABLE — once written, never modified. New data goes to new files.")

In [ ]:
# Peek inside the WAL
wal_path = os.path.join(db_path, "wal.log")
entries = WAL.replay(wal_path)
print(f"WAL contains {len(entries)} entries (only from keys not yet flushed to SSTable)")
for e in entries[:5]:
    print(f"  op={e['op']}, key={e['key']}, value={str(e['value'])[:40]}...")

print("\n→ WAL only holds entries since the last flush.")
print("→ After a flush, the WAL is truncated — those entries are now safely in an SSTable.")
print("→ This is the KEY difference from a B-tree WAL: the LSM WAL is the PRIMARY write path,")
print("  not a safety net for in-place page updates.")

In [ ]:
db.close()

# Write throughput comparison: LSM vs simulated B-tree
import sqlite3

n = 50_000
random.seed(42)
keys = [f"key_{i:08d}" for i in random.sample(range(1_000_000), n)]
values = [f"value_{i}" for i in range(n)]

# LSM-Tree writes
lsm_path = f"{OUTPUT_DIR}/bench_lsm"
lsm = LSMTree(lsm_path, memtable_capacity=5000)
start = time.time()
for k, v in zip(keys, values):
    lsm.put(k, v)
lsm_write_time = time.time() - start
lsm.close()

# B-tree writes (SQLite with indexes)
btree_path = f"{OUTPUT_DIR}/bench_btree.db"
conn = sqlite3.connect(btree_path)
c = conn.cursor()
c.execute("CREATE TABLE t (key TEXT PRIMARY KEY, value TEXT)")
c.execute("CREATE INDEX idx_key ON t(key)")
start = time.time()
c.executemany("INSERT INTO t VALUES (?,?)", zip(keys, values))
conn.commit()
btree_write_time = time.time() - start
conn.close()

print(f"Write throughput ({n:,} random keys):")
print(f"  LSM-Tree:  {lsm_write_time:.3f}s  ({n/lsm_write_time:>10,.0f} writes/sec)")
print(f"  B+Tree:    {btree_write_time:.3f}s  ({n/btree_write_time:>10,.0f} writes/sec)")
print(f"\n→ LSM writes are sequential appends: WAL + memtable insert.")
print(f"→ B-tree writes must find the right page, insert in order, and potentially split.")

---
## Part 2: The Read Path — The Price You Pay

Reads in an LSM-Tree are more expensive because data is spread across multiple places:

```
get(key) ──→ Memtable?    ──(miss)──→ SSTable L0?  ──(miss)──→ SSTable L1? ──→ ...
              (newest)                  (newer)                  (older)
              O(log N)                  O(log N) each            O(log N) each
```

**Read amplification** = number of places you must check before finding (or not finding) the key.

Bloom filters help: they can tell you "this SSTable definitely does NOT contain your key"
with zero disk I/O, avoiding pointless searches.

In [ ]:
# Reopen the demo DB and observe read behavior
db = LSMTree(f"{OUTPUT_DIR}/demo_db", memtable_capacity=10)
# Re-insert some data to have a populated state
for i in range(1, 36):
    db.put(f"user_{i:04d}", f'{{"name": "User {i}", "score": {random.randint(1, 100)}}}')

# Visualize current state
snap = get_lsm_snapshot(db)
path = render_lsm_state(**snap, title="LSM-Tree state before reads",
                         filename="read_state", output_dir=OUTPUT_DIR)
if path.endswith(".png"):
    display(Image(filename=path))

print(f"\n{db.describe()}\n")

# Read keys from different locations
test_keys = [
    ("user_0033", "should be in memtable (recent)"),
    ("user_0005", "should be in an SSTable (old)"),
    ("user_9999", "does not exist"),
]

print(f"{'Key':<15} {'Found In':<25} {'SSTables Checked':>17} {'Bloom Skipped':>14} {'Comparisons':>12}")
print("-" * 88)

for key, description in test_keys:
    value, stats = db.get(key)
    found_in = stats['found_in'] or 'NOT FOUND'
    print(f"{key:<15} {found_in:<25} {stats['sstables_checked']:>17} "
          f"{stats['bloom_skipped']:>14} {stats['total_comparisons']:>12}")
    print(f"  → {description}")

print(f"\n→ Memtable hit = instant (in-memory). No disk I/O.")
print(f"→ SSTable hit = must scan file(s). Bloom filter saves us from checking irrelevant SSTables.")
print(f"→ NOT FOUND = worst case: must check memtable + ALL SSTables before giving up.")

In [ ]:
db.close()

# Bloom filter deep-dive
print("=== Bloom Filter Internals ===")

bf = BloomFilter(capacity=1000, fp_rate=0.01)
# Add 1000 keys
for i in range(1000):
    bf.add(f"key_{i}")

# Test: all added keys should return True
false_negatives = sum(1 for i in range(1000) if not bf.might_contain(f"key_{i}"))
print(f"False negatives (should be 0): {false_negatives}")

# Test: keys NOT added — some will falsely return True
false_positives = sum(1 for i in range(1000, 11000) if bf.might_contain(f"key_{i}"))
total_checks = 10000
fp_rate_actual = false_positives / total_checks

print(f"\nBloom filter stats:")
print(f"  Bit array size: {bf.size:,} bits ({bf.size // 8:,} bytes)")
print(f"  Hash functions: {bf.num_hashes}")
print(f"  Items added: {bf.items_added:,}")
print(f"  Theoretical FP rate: {bf.false_positive_rate():.4%}")
print(f"  Actual FP rate: {fp_rate_actual:.4%} ({false_positives}/{total_checks})")
print(f"\n→ Bloom filter uses ~{bf.size // 8:,} bytes to represent {bf.items_added:,} keys.")
print(f"→ It can say 'DEFINITELY NOT HERE' (true negative) — saves an entire SSTable scan.")
print(f"→ It can say 'MAYBE HERE' (might be false positive) — must verify with actual lookup.")
print(f"→ It can NEVER say 'DEFINITELY NOT HERE' when the key IS there (zero false negatives).")

In [ ]:
# Read amplification at scale: measure how many SSTables we check as data grows
import matplotlib.pyplot as plt

memtable_sizes = [100, 500, 1000, 5000]
n_keys = 50000
random.seed(42)

results = {}
for mt_size in memtable_sizes:
    db_path = f"{OUTPUT_DIR}/readamp_{mt_size}"
    db = LSMTree(db_path, memtable_capacity=mt_size)
    
    for i in range(n_keys):
        db.put(f"key_{i:08d}", f"value_{i}")
    
    # Measure read amplification for 1000 random lookups
    total_sst_checks = 0
    total_bloom_skips = 0
    sample_keys = [f"key_{random.randint(0, n_keys-1):08d}" for _ in range(1000)]
    for key in sample_keys:
        _, stats = db.get(key)
        total_sst_checks += stats['sstables_checked']
        total_bloom_skips += stats['bloom_skipped']
    
    avg_checks = total_sst_checks / 1000
    avg_skips = total_bloom_skips / 1000
    num_sstables = len(db.sstables)
    
    results[mt_size] = {
        'avg_sst_checks': avg_checks,
        'avg_bloom_skips': avg_skips,
        'num_sstables': num_sstables,
    }
    db.close()

print(f"Read amplification vs memtable size ({n_keys:,} keys, 1000 random reads):")
print(f"{'Memtable Size':>14} {'SSTables':>9} {'Avg SST Checks':>15} {'Avg Bloom Skips':>16}")
print("-" * 58)
for mt_size in memtable_sizes:
    r = results[mt_size]
    print(f"{mt_size:>14,} {r['num_sstables']:>9} {r['avg_sst_checks']:>15.1f} {r['avg_bloom_skips']:>16.1f}")

print(f"\n→ Smaller memtable = more flushes = more SSTables = higher read amplification.")
print(f"→ Bloom filters save us from checking most SSTables, but the overhead is still there.")
print(f"→ This is the fundamental LSM tradeoff: cheap writes, expensive reads.")

---
## Part 3: Compaction — Taming the Read Amplification

Without compaction, SSTables accumulate forever:
- Read amplification grows linearly with the number of SSTables
- Deleted keys (tombstones) waste disk space
- Duplicate keys across SSTables waste space

**Compaction** merges SSTables to solve all three problems:

```
Before compaction:                After compaction:

L0: [SST_1] [SST_2] [SST_3]     L0: (empty)
     ↓  overlapping ranges        L1: [SST_merged]
                                       ↓  sorted, deduped, tombstones removed
```

Two main strategies:
- **Size-tiered** (Cassandra default): merge similarly-sized SSTables. Simple, write-optimized.
- **Leveled** (RocksDB default): maintain non-overlapping ranges per level. Read-optimized.

In [ ]:
# Demonstrate compaction step by step with diagrams
db_path = f"{OUTPUT_DIR}/compact_demo"
db = LSMTree(db_path, memtable_capacity=5, l0_compaction_trigger=3)

print("Inserting keys to trigger flushes and compaction...")
print("(memtable capacity=5, compact when L0 has 3 SSTables)\n")

compact_snapshots = []
for i in range(1, 26):
    key = f"key_{i:03d}"
    result = db.put(key, f"value_{i}")
    if result['flushed'] or result['compacted']:
        events = []
        if result['flushed']: events.append("FLUSH")
        if result['compacted']: events.append("COMPACT")
        label = ' + '.join(events)
        print(f"  After inserting {key}: {label}")
        print(f"    {db.describe()}")
        snap = get_lsm_snapshot(db)
        compact_snapshots.append((key, label, snap))
        print()

# Show before/after compaction visually
for key, label, snap in compact_snapshots:
    title = f"After {key}: {label}"
    fname = f"compact_{key}_{label.replace(' + ', '_').lower()}"
    path = render_lsm_state(**snap, title=title, filename=fname, output_dir=OUTPUT_DIR)
    if path.endswith(".png"):
        display(Image(filename=path))

db.close()

print("→ Watch L0 SSTables accumulate, then get COMPACTED into a single L1 SSTable.")
print("→ Compaction reduces read amplification: fewer SSTables to check.")

In [ ]:
# Compaction removes tombstones (deleted keys)
db_path = f"{OUTPUT_DIR}/tombstone_demo"
db = LSMTree(db_path, memtable_capacity=10, l0_compaction_trigger=3)

# Insert 20 keys
for i in range(1, 21):
    db.put(f"key_{i:03d}", f"value_{i}")

# Delete half of them
for i in range(1, 11):
    db.delete(f"key_{i:03d}")

print("Before compaction:")
print(f"  {db.describe()}")
usage_before = db.disk_usage()
total_before = usage_before['total']

# Force flush + compaction
for i in range(21, 40):
    db.put(f"key_{i:03d}", f"value_{i}")

print(f"\nAfter writes trigger compaction:")
print(f"  {db.describe()}")
usage_after = db.disk_usage()
total_after = usage_after['total']

print(f"\nDisk usage: {total_before:,} → {total_after:,} bytes")
print(f"→ Compaction merged SSTables and garbage-collected tombstones.")
print(f"→ In a real system, this runs in the background continuously.")

db.close()

In [ ]:
# Write amplification: how much data do we actually write to disk vs what the user wrote?

db_path = f"{OUTPUT_DIR}/write_amp"
db = LSMTree(db_path, memtable_capacity=1000, l0_compaction_trigger=4)

n = 20_000
random.seed(42)
user_bytes = 0

for i in range(n):
    key = f"key_{i:08d}"
    value = f"value_{random.randint(1, 1000000):010d}"
    user_bytes += len(key) + len(value)
    db.put(key, value)

# Calculate actual disk usage
disk_bytes = 0
for root, dirs, files in os.walk(db_path):
    for f in files:
        disk_bytes += os.path.getsize(os.path.join(root, f))

write_amp = disk_bytes / user_bytes

print(f"Write amplification ({n:,} keys):")
print(f"  User data written:  {user_bytes:>12,} bytes")
print(f"  Actual disk usage:  {disk_bytes:>12,} bytes")
print(f"  Write amplification: {write_amp:.2f}x")
print(f"  Flushes: {db.stats['flushes']}, Compactions: {db.stats['compactions']}")
print(f"\n→ Write amplification > 1x because data is written multiple times:")
print(f"   1st write: WAL, 2nd write: SSTable flush, 3rd+ write: compaction merges")
print(f"→ This is the hidden cost of LSM-trees: cheap logical writes, but physical writes multiply.")

db.close()

---
## Part 4: Crash Recovery — The WAL Saves You

What happens when the process dies?

- **SSTables on disk**: safe — they're immutable files, fully written before being visible
- **Memtable in RAM**: LOST — it's volatile memory
- **WAL on disk**: safe — it has every write since the last flush

Recovery = replay the WAL to reconstruct the memtable.

```
                     CRASH!
                       |
  Memtable (RAM): [k1, k2, k3]  ←── GONE
  WAL (disk):     [k1, k2, k3]  ←── SAFE! Replay this.
  SSTables:       [older data]   ←── SAFE! Immutable files.
```

In [ ]:
# Simulate crash recovery
db_path = f"{OUTPUT_DIR}/crash_demo"

# Phase 1: Write data, then "crash" (don't close cleanly)
print("=== Phase 1: Writing data ===")
db = LSMTree(db_path, memtable_capacity=20)

for i in range(1, 51):
    db.put(f"user_{i:04d}", f'{{"email": "user{i}@company.com", "active": true}}')

memtable_keys = len(db.memtable)
sstable_count = len(db.sstables)
print(f"  Written 50 keys")
print(f"  Memtable: {memtable_keys} keys (IN MEMORY — will be lost on crash)")
print(f"  SSTables: {sstable_count} files (ON DISK — safe)")
print(f"  WAL: {db.wal.write_count} entries (ON DISK — safe)")

# Verify we can read everything before crash
val_before, _ = db.get("user_0045")
print(f"  Read user_0045 before crash: {'FOUND' if val_before else 'NOT FOUND'}")

# SIMULATE CRASH: just abandon the object without closing
# (In reality, the process would die and memtable would be lost)
print("\n  💥 CRASH! Process dies. Memtable is lost.")
del db  # memtable is gone

In [ ]:
# Phase 2: Recovery — reconstruct from WAL + SSTables
print("=== Phase 2: Recovery ===")
print("  Discovering SSTables on disk...")
print("  Replaying WAL to reconstruct memtable...")

db_recovered, recovery_stats = LSMTree.recover(db_path, memtable_capacity=20)

print(f"\n  Recovery stats:")
print(f"    SSTables found: {recovery_stats['sstables_found']}")
print(f"    WAL entries replayed: {recovery_stats['wal_entries_replayed']}")
print(f"    Keys recovered to memtable: {recovery_stats['keys_recovered']}")
print(f"    Recovery time: {recovery_stats['recovery_time_ms']:.2f} ms")

# Verify data integrity
val_after, _ = db_recovered.get("user_0045")
print(f"\n  Read user_0045 after recovery: {'FOUND' if val_after else 'NOT FOUND'}")
print(f"  Data matches: {val_before == val_after}")

# Verify ALL keys survived
recovered_count = 0
for i in range(1, 51):
    val, _ = db_recovered.get(f"user_{i:04d}")
    if val is not None:
        recovered_count += 1

print(f"  Keys recovered: {recovered_count}/50")
print(f"\n→ ALL data survived the crash!")
print(f"→ SSTables were already on disk. WAL entries reconstructed the memtable.")
print(f"→ Recovery is FAST: just replay the WAL (sequential read). No tree rebuilding needed.")

db_recovered.close()

In [ ]:
# How fast is recovery vs data size?
recovery_times = []
data_sizes = [100, 500, 1000, 5000, 10000]

for n in data_sizes:
    db_path = f"{OUTPUT_DIR}/recovery_bench_{n}"
    db = LSMTree(db_path, memtable_capacity=n + 1)  # never flush — all in memtable+WAL
    for i in range(n):
        db.put(f"key_{i:08d}", f"value_{i:020d}")
    del db  # "crash"
    
    _, stats = LSMTree.recover(db_path, memtable_capacity=n + 1)
    recovery_times.append(stats['recovery_time_ms'])

print("Recovery time vs WAL size:")
print(f"{'WAL Entries':>12} {'Recovery Time':>14}")
print("-" * 28)
for n, t in zip(data_sizes, recovery_times):
    print(f"{n:>12,} {t:>12.2f} ms")

print(f"\n→ Recovery is linear in WAL size — just sequential read + memtable inserts.")
print(f"→ This is why LSM-trees keep the WAL small: flush often → shorter recovery.")

---
## Part 4b: Sequential vs Random Writes — Why LSM Loves Ordered Keys

B+Trees struggle with random inserts (must find the right page, split on overflow).
LSM-Trees don't care — every write is just an append to the WAL + memtable insert.

But what about **sequential** (monotonically increasing) keys like auto-increment IDs or timestamps?
LSM-Trees are *especially* fast here. Let's see why.

In [ ]:
# Sequential vs Random writes: LSM-Tree
import matplotlib.pyplot as plt
import sqlite3

n = 50_000
random.seed(42)
random_keys = [f"key_{i:08d}" for i in random.sample(range(1_000_000), n)]
sequential_keys = [f"key_{i:08d}" for i in range(n)]  # monotonically increasing

def lsm_write_bench(keys, label):
    db_path = f"{OUTPUT_DIR}/seqrnd_lsm_{label}"
    db = LSMTree(db_path, memtable_capacity=5000, l0_compaction_trigger=4)
    start = time.time()
    for k in keys:
        db.put(k, f"value_{k}")
    elapsed = time.time() - start
    flushes = db.stats['flushes']
    compactions = db.stats['compactions']
    db.close()
    return elapsed, flushes, compactions

def btree_write_bench(keys, label):
    db_path = f"{OUTPUT_DIR}/seqrnd_btree_{label}.db"
    conn = sqlite3.connect(db_path)
    c = conn.cursor()
    c.execute("CREATE TABLE t (key TEXT PRIMARY KEY, value TEXT)")
    start = time.time()
    c.executemany("INSERT INTO t VALUES (?,?)", [(k, f"value_{k}") for k in keys])
    conn.commit()
    elapsed = time.time() - start
    conn.close()
    return elapsed

lsm_seq_time, lsm_seq_flush, lsm_seq_compact = lsm_write_bench(sequential_keys, "seq")
lsm_rnd_time, lsm_rnd_flush, lsm_rnd_compact = lsm_write_bench(random_keys, "rnd")
btree_seq_time = btree_write_bench(sequential_keys, "seq")
btree_rnd_time = btree_write_bench(random_keys, "rnd")

print(f"Write throughput ({n:,} keys):")
print(f"{'':>14} {'Sequential':>14} {'Random':>14} {'Seq/Rnd':>10}")
print("-" * 55)
print(f"{'LSM-Tree':>14} {lsm_seq_time:>12.3f}s {lsm_rnd_time:>12.3f}s {lsm_rnd_time/lsm_seq_time:>9.2f}x")
print(f"{'B+Tree':>14} {btree_seq_time:>12.3f}s {btree_rnd_time:>12.3f}s {btree_rnd_time/btree_seq_time:>9.2f}x")
print(f"\nLSM flushes:     seq={lsm_seq_flush}, rnd={lsm_rnd_flush}")
print(f"LSM compactions: seq={lsm_seq_compact}, rnd={lsm_rnd_compact}")

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
labels = ['LSM Sequential', 'LSM Random', 'B+Tree Sequential', 'B+Tree Random']
times = [lsm_seq_time, lsm_rnd_time, btree_seq_time, btree_rnd_time]
colors = ['#2196F3', '#90CAF9', '#F44336', '#EF9A9A']
bars = ax.bar(labels, [n/t for t in times], color=colors)
ax.set_ylabel('Writes/sec')
ax.set_title(f'Sequential vs Random Writes ({n:,} keys)')
ax.grid(True, alpha=0.3, axis='y')
for bar, t in zip(bars, times):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height(),
            f'{n/t:,.0f}/s', ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/seq_vs_random.png', dpi=150)
plt.show()

print("\n→ LSM-Tree: sequential and random are similar — both are just WAL appends + memtable inserts.")
print("  The memtable sorts everything regardless of input order.")
print("\n→ B+Tree: sequential is faster because inserts always go to the rightmost leaf (no searching).")
print("  Random inserts must traverse the tree to find the right page each time.")

### What RocksDB does to make sequential writes even faster

Our pure-Python LSM shows the concept. RocksDB (used in production at Meta, Netflix, Uber) adds:

1. **Memtable = Skip List** — O(log N) insert with lock-free concurrent writes. Sequential keys hit the tail of the skip list — cache-friendly, near O(1).

2. **WAL = Direct I/O + group commit** — batches multiple WAL appends into a single `fsync`. Sequential writes batch better because they arrive in bursts.

3. **SSTable = Block-based format** — sequential keys compress extremely well (shared prefixes via prefix compression). A 100MB SSTable of sequential keys might compress to 20MB.

4. **Compaction = Non-overlapping key ranges at L1+** — sequential inserts produce SSTables with non-overlapping ranges, so L0→L1 compaction is trivial (just move the file, no merge needed). This is called a "trivial move" in RocksDB.

5. **Rate limiter + write stalls** — RocksDB throttles writes if compaction falls behind, preventing unbounded L0 growth. Sequential workloads rarely trigger stalls because compaction is simpler.

**Bottom line**: LSM-Trees don't care about key order for correctness, but sequential keys are a *performance gift* — better compression, simpler compaction, and cache-friendly memtable access.

---
## Part 5: B+Tree vs LSM-Tree — Head to Head

Now let's put them side by side and see the tradeoffs concretely.

In [ ]:
# Head-to-head benchmark

def bench_lsm(n_rows, mt_size=5000):
    db_path = f"{OUTPUT_DIR}/h2h_lsm_{n_rows}"
    db = LSMTree(db_path, memtable_capacity=mt_size, l0_compaction_trigger=4)
    random.seed(42)
    keys = [f"key_{i:08d}" for i in random.sample(range(1_000_000), n_rows)]
    
    # Writes
    start = time.time()
    for k in keys:
        db.put(k, f"value_{k}")
    write_time = time.time() - start
    
    # Point reads (random)
    sample = random.sample(keys, min(1000, n_rows))
    start = time.time()
    for k in sample:
        db.get(k)
    read_time = (time.time() - start) / len(sample)
    
    db.close()
    return write_time, read_time

def bench_btree(n_rows):
    db_path = f"{OUTPUT_DIR}/h2h_btree_{n_rows}.db"
    conn = sqlite3.connect(db_path)
    c = conn.cursor()
    c.execute("CREATE TABLE t (key TEXT PRIMARY KEY, value TEXT)")
    random.seed(42)
    keys = [f"key_{i:08d}" for i in random.sample(range(1_000_000), n_rows)]
    
    # Writes
    start = time.time()
    c.executemany("INSERT INTO t VALUES (?,?)", [(k, f"value_{k}") for k in keys])
    conn.commit()
    write_time = time.time() - start
    
    # Point reads
    sample = random.sample(keys, min(1000, n_rows))
    start = time.time()
    for k in sample:
        c.execute("SELECT value FROM t WHERE key = ?", (k,)).fetchone()
    read_time = (time.time() - start) / len(sample)
    
    conn.close()
    return write_time, read_time

sizes = [1000, 5000, 10000, 50000]
lsm_results = {}
btree_results = {}

print("Running head-to-head benchmarks...")
for n in sizes:
    print(f"  {n:>7,} rows...", end=" ", flush=True)
    lsm_results[n] = bench_lsm(n)
    btree_results[n] = bench_btree(n)
    print("done")

print(f"\n{'Rows':>8} {'LSM Write':>12} {'B+T Write':>12} {'LSM Read':>12} {'B+T Read':>12}")
print("-" * 60)
for n in sizes:
    lw, lr = lsm_results[n]
    bw, br = btree_results[n]
    print(f"{n:>8,} {lw:>10.3f}s {bw:>10.3f}s {lr*1000:>10.3f}ms {br*1000:>10.3f}ms")

In [ ]:
# Visualize the tradeoff
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Write throughput
lsm_wps = [n / lsm_results[n][0] for n in sizes]
btree_wps = [n / btree_results[n][0] for n in sizes]
ax1.plot(sizes, lsm_wps, 'o-', color='blue', linewidth=2, label='LSM-Tree')
ax1.plot(sizes, btree_wps, 'o-', color='red', linewidth=2, label='B+Tree (SQLite)')
ax1.set_xlabel('Number of Rows')
ax1.set_ylabel('Writes/sec')
ax1.set_title('Write Throughput')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_xscale('log')

# Read latency
lsm_rms = [lsm_results[n][1] * 1000 for n in sizes]
btree_rms = [btree_results[n][1] * 1000 for n in sizes]
ax2.plot(sizes, lsm_rms, 'o-', color='blue', linewidth=2, label='LSM-Tree')
ax2.plot(sizes, btree_rms, 'o-', color='red', linewidth=2, label='B+Tree (SQLite)')
ax2.set_xlabel('Number of Rows')
ax2.set_ylabel('Read Latency (ms)')
ax2.set_title('Point Read Latency')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_xscale('log')
ax2.set_yscale('log')

plt.suptitle('B+Tree vs LSM-Tree: The Fundamental Tradeoff', fontsize=14)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/btree_vs_lsm.png', dpi=150)
plt.show()

print("\nThe tradeoff visualized:")
print("  LEFT:  LSM-Tree wins on writes (sequential I/O, no rebalancing)")
print("  RIGHT: B+Tree wins on reads (single sorted structure, O(log N))")

---
## Summary: The Three Amplification Factors

Every storage engine must balance three types of amplification:

| Factor | B+Tree | LSM-Tree |
|--------|--------|----------|
| **Write amplification** | High (in-place updates, splits) | Medium (WAL + flush + compaction) |
| **Read amplification** | Low (one sorted structure) | High (check memtable + multiple SSTables) |
| **Space amplification** | Low (~1x) | Medium-High (duplicates across levels until compacted) |

### When to use which?

| Workload | Best fit | Why | Real systems |
|----------|----------|-----|-------------|
| **Read-heavy OLTP** | B+Tree | Fast point lookups, range scans | PostgreSQL, MySQL/InnoDB |
| **Write-heavy ingestion** | LSM-Tree | Sequential writes, no rebalancing | RocksDB, Cassandra, HBase |
| **Time-series data** | LSM-Tree | Append-mostly, rarely read old data | InfluxDB, TimescaleDB |
| **Mixed OLTP** | B+Tree | Predictable read latency matters | Most traditional RDBMS |
| **Key-value store** | LSM-Tree | High write throughput, eventual reads | LevelDB, RocksDB, BadgerDB |

### The mental model

```
B+Tree:   "Sort on write, read for free"
          Every write maintains sorted order → reads are instant.

LSM-Tree: "Write for free, sort on read"
          Writes just append → reads must search multiple sorted runs.
```

### What's next: Columnar Formats (Parquet)

Both B+Trees and LSM-Trees store data **row by row** — good for OLTP (fetch one row at a time).
For analytics (scan millions of rows, few columns), **columnar** formats like Parquet flip the layout
entirely: store each column separately for massive compression and scan speed.

In [ ]:
# Final cleanup
import glob
print("Lab complete! Output files:")
total = 0
for root, dirs, files in os.walk(OUTPUT_DIR):
    for f in files:
        path = os.path.join(root, f)
        size = os.path.getsize(path)
        total += size
dirs_count = sum(1 for _ in os.scandir(OUTPUT_DIR) if _.is_dir())
files_count = sum(len(files) for _, _, files in os.walk(OUTPUT_DIR))
print(f"  {dirs_count} directories, {files_count} files, {total:,} bytes total")
print(f"  All contained in {OUTPUT_DIR}/")